In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from chemotools.derivative import SavitzkyGolay
from typing import Tuple

# ============================================================================
# CONFIGURAÇÕES GLOBAIS
# ============================================================================
KEEP_INTERMEDIATE = False  # Se True, mantém arquivos intermediários
GENERATE_PLOTS = True      # Se True, gera gráficos de cada etapa

# ============================================================================
# ETAPA 1: MULTIPLICATIVE SIGNAL CORRECTION (MSC)
# ============================================================================
def apply_msc(output_dir: str, folder_name: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Aplica Multiplicative Signal Correction (MSC) para corrigir variações
    de espalhamento e iluminação.
    
    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: (dados_antes, dados_depois)
    """
    input_path = os.path.join(output_dir, f"{folder_name}_roi.csv")
    output_path = os.path.join(output_dir, f"{folder_name}_msc_signal.csv")
    
    print("\n" + "="*70)
    print("ETAPA 1: MULTIPLICATIVE SIGNAL CORRECTION (MSC)")
    print("="*70)
    
    if not os.path.exists(input_path):
        print(f"⚠️  {input_path} não encontrado. Criando saída vazia.")
        pd.DataFrame().to_csv(output_path, index=False)
        return pd.DataFrame(), pd.DataFrame()
    
    df_before = pd.read_csv(input_path)
    if df_before.empty:
        print(f"⚠️  {input_path} vazio. Criando saída vazia.")
        pd.DataFrame().to_csv(output_path, index=False)
        return df_before, pd.DataFrame()
    
    data = df_before.values
    ref = np.mean(data, axis=0)
    
    # Aplicar MSC com cálculo otimizado
    corrected = []
    for s in data:
        a, b = np.polyfit(ref, s, 1)  # a=slope, b=intercept
        s_corrected = (s - b) / a
        corrected.append(s_corrected)
    
    df_after = pd.DataFrame(corrected, columns=df_before.columns)
    df_after.to_csv(output_path, index=False)
    
    print(f"✅ MSC aplicado com sucesso")
    print(f"   📊 Amostras: {df_before.shape[0]}")
    print(f"   📊 Bandas: {df_before.shape[1]}")
    print(f"   📊 Range antes: [{data.min():.4f}, {data.max():.4f}]")
    print(f"   📊 Range depois: [{df_after.values.min():.4f}, {df_after.values.max():.4f}]")
    print(f"   💾 Arquivo salvo: {output_path}")
    
    if not KEEP_INTERMEDIATE and os.path.exists(input_path):
        os.remove(input_path)
        print(f"   🧹 Arquivo intermediário removido: {input_path}")
    
    return df_before, df_after


# ============================================================================
# ETAPA 2: TRANSFORMAÇÃO LOGARÍTMICA Log(1/R)
# ============================================================================
def apply_log_transform(output_dir: str, folder_name: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Aplica transformação logarítmica log(1/R) = -log(R) para converter
    reflectância em pseudo-absorbância.
    
    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: (dados_antes, dados_depois)
    """
    input_path = os.path.join(output_dir, f"{folder_name}_msc_signal.csv")
    output_path = os.path.join(output_dir, f"{folder_name}_log.csv")
    
    print("\n" + "="*70)
    print("ETAPA 2: TRANSFORMAÇÃO LOGARÍTMICA log(1/R)")
    print("="*70)
    
    if not os.path.exists(input_path):
        print(f"⚠️  {input_path} não encontrado. Criando saída vazia.")
        pd.DataFrame().to_csv(output_path, index=False)
        return pd.DataFrame(), pd.DataFrame()
    
    df_before = pd.read_csv(input_path)
    if df_before.empty:
        print(f"⚠️  {input_path} vazio. Criando saída vazia.")
        pd.DataFrame().to_csv(output_path, index=False)
        return df_before, pd.DataFrame()
    
    data = df_before.values
    
    # Substituir valores <= 0 para evitar log(0) ou log(negativo)
    n_invalid = (data <= 0).sum()
    if n_invalid > 0:
        print(f"   ⚠️  {n_invalid} valores ≤ 0 substituídos por 1e-6")
    data[data <= 0] = 1e-6
    
    # Aplicar transformação: -log(R) = log(1/R)
    transformed = -np.log(data)
    
    df_after = pd.DataFrame(transformed, columns=df_before.columns)
    df_after.to_csv(output_path, index=False)
    
    print(f"✅ Transformação log(1/R) aplicada com sucesso")
    print(f"   📊 Range antes: [{df_before.values.min():.4f}, {df_before.values.max():.4f}]")
    print(f"   📊 Range depois: [{transformed.min():.4f}, {transformed.max():.4f}]")
    print(f"   💾 Arquivo salvo: {output_path}")
    
    if not KEEP_INTERMEDIATE and os.path.exists(input_path):
        os.remove(input_path)
        print(f"   🧹 Arquivo intermediário removido: {input_path}")
    
    return df_before, df_after


# ============================================================================
# ETAPA 3: DERIVADA DE SAVITZKY-GOLAY (1ª ORDEM)
# ============================================================================
def apply_derivative(output_dir: str, folder_name: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Aplica derivada de primeira ordem usando filtro Savitzky-Golay
    (janela=15, polinômio=2).
    
    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: (dados_antes, dados_depois)
    """
    input_path = os.path.join(output_dir, f"{folder_name}_log.csv")
    output_path = os.path.join(output_dir, f"{folder_name}_derivada.csv")
    
    print("\n" + "="*70)
    print("ETAPA 3: DERIVADA DE SAVITZKY-GOLAY (1ª ORDEM)")
    print("="*70)
    
    if not os.path.exists(input_path):
        print(f"⚠️  {input_path} não encontrado. Criando saída vazia.")
        pd.DataFrame().to_csv(output_path, index=False)
        return pd.DataFrame(), pd.DataFrame()
    
    df_before = pd.read_csv(input_path)
    if df_before.empty:
        print(f"⚠️  {input_path} vazio. Criando saída vazia.")
        pd.DataFrame().to_csv(output_path, index=False)
        return df_before, pd.DataFrame()
    
    # Aplicar Savitzky-Golay com derivada de 1ª ordem
    # Nota: Verifique a documentação da biblioteca chemotools
    # Se não tiver parâmetro 'deriv', considere usar scipy.signal.savgol_filter
    try:
        deriv = SavitzkyGolay(window_size=15, polynomial_order=2, deriv=1)
    except TypeError:
        # Fallback se a biblioteca não suportar parâmetro deriv
        print("   ⚠️  Usando SavitzkyGolay sem parâmetro 'deriv' explícito")
        deriv = SavitzkyGolay(window_size=15, polynomial_order=2)
    
    transformed = deriv.fit_transform(df_before.values)
    
    df_after = pd.DataFrame(transformed, columns=df_before.columns)
    df_after.to_csv(output_path, index=False)
    
    print(f"✅ Derivada de 1ª ordem aplicada com sucesso")
    print(f"   📊 Janela: 15 pontos")
    print(f"   📊 Polinômio: grau 2")
    print(f"   📊 Range antes: [{df_before.values.min():.4f}, {df_before.values.max():.4f}]")
    print(f"   📊 Range depois: [{transformed.min():.4f}, {transformed.max():.4f}]")
    print(f"   💾 Arquivo salvo: {output_path}")
    
    if not KEEP_INTERMEDIATE and os.path.exists(input_path):
        os.remove(input_path)
        print(f"   🧹 Arquivo intermediário removido: {input_path}")
    
    return df_before, df_after


# ============================================================================
# ETAPA 4: FILTRAGEM DE BANDAS RUIDOSAS
# ============================================================================
def apply_noise_filtering(output_dir: str, folder_name: str) -> Tuple[pd.DataFrame, pd.DataFrame, list]:
    """
    Remove bandas com baixa variância (indicativo de ruído).
    Critérios:
    - Extremidades (10% cada lado): remove se variância < 5% do máximo
    - Centro (80%): remove se variância < 1% do máximo
    
    Returns:
        Tuple[pd.DataFrame, pd.DataFrame, list]: (dados_antes, dados_depois, bandas_removidas)
    """
    input_path = os.path.join(output_dir, f"{folder_name}_derivada.csv")
    output_path = os.path.join(output_dir, f"{folder_name}_filtrado.csv")
    
    print("\n" + "="*70)
    print("ETAPA 4: FILTRAGEM DE BANDAS RUIDOSAS")
    print("="*70)
    
    if not os.path.exists(input_path):
        print(f"⚠️  {input_path} não encontrado. Criando saída vazia.")
        pd.DataFrame().to_csv(output_path, index=False)
        return pd.DataFrame(), pd.DataFrame(), []
    
    df_before = pd.read_csv(input_path)
    if df_before.empty:
        print(f"⚠️  {input_path} vazio. Criando saída vazia.")
        pd.DataFrame().to_csv(output_path, index=False)
        return df_before, pd.DataFrame(), []
    
    data = df_before.values
    variancias = np.var(data, axis=0)
    n_ext = int(data.shape[1] * 0.10)
    lim_ext = 0.05 * np.max(variancias)
    lim_centro = 0.01 * np.max(variancias)
    
    # Identificar bandas problemáticas
    bandas_problematicas = []
    
    # Extremidades
    for i in range(n_ext):
        if variancias[i] < lim_ext:
            bandas_problematicas.append(i)
        if variancias[-(i+1)] < lim_ext:
            bandas_problematicas.append(data.shape[1] - (i+1))
    
    # Centro
    for i in range(n_ext, data.shape[1] - n_ext):
        if variancias[i] < lim_centro:
            bandas_problematicas.append(i)
    
    bandas_problematicas = sorted(set(bandas_problematicas))
    
    # Remover bandas
    data_filtrada = np.delete(data, bandas_problematicas, axis=1)
    colunas_filtradas = [c for i, c in enumerate(df_before.columns) if i not in bandas_problematicas]
    
    df_after = pd.DataFrame(data_filtrada, columns=colunas_filtradas)
    df_after.to_csv(output_path, index=False)
    
    print(f"✅ Filtragem de bandas ruidosas concluída")
    print(f"   📊 Total de bandas: {data.shape[1]}")
    print(f"   📊 Bandas removidas: {len(bandas_problematicas)} ({100*len(bandas_problematicas)/data.shape[1]:.1f}%)")
    print(f"   📊 Bandas restantes: {data_filtrada.shape[1]}")
    print(f"   📊 Variância mín/máx: [{np.min(variancias):.2e}, {np.max(variancias):.2e}]")
    print(f"   📊 Limiar extremidades: {lim_ext:.2e}")
    print(f"   📊 Limiar centro: {lim_centro:.2e}")
    print(f"   💾 Arquivo salvo: {output_path}")
    
    if not KEEP_INTERMEDIATE and os.path.exists(input_path):
        os.remove(input_path)
        print(f"   🧹 Arquivo intermediário removido: {input_path}")
    
    return df_before, df_after, bandas_problematicas


# ============================================================================
# VISUALIZAÇÕES
# ============================================================================

def plot_msc_comparison(df_before: pd.DataFrame, df_after: pd.DataFrame, 
                        output_dir: str, folder_name: str):
    """Gráfico comparativo MSC: Antes vs Depois"""
    if df_before.empty or df_after.empty:
        print("   ⚠️  Dados vazios, pulando visualização MSC")
        return
    
    plt.figure(figsize=(14, 6))
    
    bandas_numeros = np.arange(df_before.shape[1])
    n_samples = min(500, df_before.shape[0])
    
    # Plotar amostras individuais (transparentes)
    for i in range(n_samples):
        if i == 0:
            plt.plot(bandas_numeros, df_before.iloc[i, :], color='lightblue', 
                    alpha=0.05, linewidth=0.5, label='Amostras Antes MSC')
            plt.plot(bandas_numeros, df_after.iloc[i, :], color='lightcoral', 
                    alpha=0.05, linewidth=0.5, label='Amostras Depois MSC')
        else:
            plt.plot(bandas_numeros, df_before.iloc[i, :], color='lightblue', 
                    alpha=0.05, linewidth=0.5)
            plt.plot(bandas_numeros, df_after.iloc[i, :], color='lightcoral', 
                    alpha=0.05, linewidth=0.5)
    
    # Plotar médias
    media_antes = df_before.mean(axis=0)
    media_depois = df_after.mean(axis=0)
    
    plt.plot(bandas_numeros, media_antes, color='blue', linewidth=3, 
            label='Média Antes MSC', zorder=10)
    plt.plot(bandas_numeros, media_depois, color='red', linewidth=3, 
            label='Média Depois MSC', zorder=10)
    
    plt.xlabel('Número da Banda', fontsize=12)
    plt.ylabel('Intensidade Normalizada (MSC)', fontsize=12)
    plt.title('Etapa 1: Multiplicative Signal Correction (MSC)', fontsize=14, fontweight='bold')
    plt.legend(loc='best', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.xticks(bandas_numeros[::len(bandas_numeros)//10])
    
    plt.tight_layout()
    plot_path = os.path.join(output_dir, f"{folder_name}_01_msc.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   📊 Gráfico salvo: {plot_path}")


def plot_log_comparison(df_before: pd.DataFrame, df_after: pd.DataFrame,
                       output_dir: str, folder_name: str):
    """Gráfico comparativo Log(1/R): Antes vs Depois"""
    if df_before.empty or df_after.empty:
        print("   ⚠️  Dados vazios, pulando visualização Log")
        return
    
    plt.figure(figsize=(14, 6))
    
    bandas_numeros = np.arange(df_before.shape[1])
    media_antes = df_before.mean(axis=0)
    media_depois = df_after.mean(axis=0)
    
    plt.plot(bandas_numeros, media_antes, label='Média Antes log(1/R)', 
            linewidth=2.5, color='blue', alpha=0.8)
    plt.plot(bandas_numeros, media_depois, label='Média Depois log(1/R)', 
            linewidth=2.5, color='red', alpha=0.8)
    
    # Adicionar linha de referência em y=0
    plt.axhline(0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
    
    plt.xlabel('Número da Banda', fontsize=12)
    plt.ylabel('Intensidade', fontsize=12)
    plt.title('Etapa 2: Transformação Logarítmica log(1/R)', fontsize=14, fontweight='bold')
    plt.legend(loc='best', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.xticks(bandas_numeros[::len(bandas_numeros)//10])
    
    # Adicionar informação de range
    info_text = (f"Antes: [{media_antes.min():.3f}, {media_antes.max():.3f}]\n"
                 f"Depois: [{media_depois.min():.3f}, {media_depois.max():.3f}]")
    plt.text(0.02, 0.98, info_text, transform=plt.gca().transAxes, 
            verticalalignment='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
    
    plt.tight_layout()
    plot_path = os.path.join(output_dir, f"{folder_name}_02_log.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   📊 Gráfico salvo: {plot_path}")


def plot_derivative_comparison(df_before: pd.DataFrame, df_after: pd.DataFrame,
                               output_dir: str, folder_name: str):
    """Gráfico comparativo Derivada: Original vs Derivada com dois eixos Y"""
    if df_before.empty or df_after.empty:
        print("   ⚠️  Dados vazios, pulando visualização Derivada")
        return
    
    fig, ax1 = plt.subplots(figsize=(14, 6))
    
    bandas_numeros = np.arange(df_before.shape[1])
    media_original = df_before.mean(axis=0)
    media_derivada = df_after.mean(axis=0)
    
    # Eixo Y esquerdo: Original (log)
    color1 = 'tab:blue'
    ax1.set_xlabel('Número da Banda', fontsize=12)
    ax1.set_ylabel('log(1/R)', color=color1, fontsize=12)
    ax1.plot(bandas_numeros, media_original, color=color1, linewidth=2.5, 
            label='Média Original [log(1/R)]', alpha=0.8)
    ax1.tick_params(axis='y', labelcolor=color1)
    ax1.grid(True, alpha=0.3)
    
    # Eixo Y direito: Derivada
    ax2 = ax1.twinx()
    color2 = 'tab:red'
    ax2.set_ylabel('d[log(1/R)]/dλ', color=color2, fontsize=12)
    ax2.plot(bandas_numeros, media_derivada, color=color2, linewidth=2.5, 
            label='Média Derivada 1ª ordem', alpha=0.8)
    ax2.tick_params(axis='y', labelcolor=color2)
    ax2.axhline(0, color='gray', linestyle='--', alpha=0.3, linewidth=1)
    
    plt.title('Etapa 3: Derivada de Savitzky-Golay (1ª ordem, janela=15, polinômio=2)', 
             fontsize=14, fontweight='bold')
    
    # Combinar legendas
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=10)
    
    fig.tight_layout()
    plot_path = os.path.join(output_dir, f"{folder_name}_03_derivada.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   📊 Gráfico salvo: {plot_path}")


def plot_filtering_analysis(df_before: pd.DataFrame, df_after: pd.DataFrame,
                            bandas_removidas: list, output_dir: str, folder_name: str):
    """Gráfico duplo: Análise de variância + Comparação espectral"""
    if df_before.empty or df_after.empty:
        print("   ⚠️  Dados vazios, pulando visualização Filtragem")
        return
    
    data = df_before.values
    variancias = np.var(data, axis=0)
    n_ext = int(data.shape[1] * 0.10)
    lim_ext = 0.05 * np.max(variancias)
    lim_centro = 0.01 * np.max(variancias)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # ===== SUBPLOT 1: Análise de Variância =====
    bandas_numeros = np.arange(len(variancias))
    
    ax1.plot(bandas_numeros, variancias, 'b-', alpha=0.7, linewidth=1.5, 
            label='Variância por banda')
    
    # Limiares
    ax1.axhline(y=lim_ext, color='darkred', linestyle='--', linewidth=2,
               label=f'Limiar extremos: {lim_ext:.2e}')
    ax1.axhline(y=lim_centro, color='orange', linestyle='--', linewidth=2,
               label=f'Limiar centro: {lim_centro:.2e}')
    
    # Regiões de extremidades
    ax1.axvspan(0, n_ext-1, alpha=0.15, color='red', label='Extremidades (10%)')
    ax1.axvspan(data.shape[1]-n_ext, data.shape[1]-1, alpha=0.15, color='red')
    
    # Bandas removidas
    if bandas_removidas:
        ax1.scatter(bandas_removidas, [variancias[i] for i in bandas_removidas], 
                   color='red', s=30, zorder=5, label=f'Bandas removidas ({len(bandas_removidas)})')
    
    ax1.set_xlabel('Número da Banda', fontsize=12)
    ax1.set_ylabel('Variância', fontsize=12)
    ax1.set_title('Análise de Variância - Detecção de Bandas Ruidosas', 
                 fontsize=12, fontweight='bold')
    ax1.legend(loc='best', fontsize=9)
    ax1.grid(True, alpha=0.3)
    ax1.set_yscale('log')  # Escala logarítmica para melhor visualização
    
    # ===== SUBPLOT 2: Comparação Espectral =====
    media_antes = np.mean(data, axis=0)
    media_depois = np.mean(df_after.values, axis=0)
    
    # Criar versão alinhada com NaN nas bandas removidas
    media_depois_alinhada = np.full_like(media_antes, np.nan)
    indices_uteis = [i for i in range(len(media_antes)) if i not in bandas_removidas]
    media_depois_alinhada[indices_uteis] = media_depois
    
    ax2.plot(bandas_numeros, media_antes, 'b-', alpha=0.7, linewidth=2,
            label='Média Antes da filtragem')
    ax2.plot(bandas_numeros, media_depois_alinhada, 'r-', linewidth=2.5,
            label='Média Depois da filtragem')
    
    # Marcar bandas removidas
    if bandas_removidas:
        for banda in bandas_removidas:
            ax2.axvline(banda, color='red', alpha=0.1, linewidth=0.5)
    
    ax2.set_xlabel('Número da Banda', fontsize=12)
    ax2.set_ylabel('Valor da Derivada', fontsize=12)
    ax2.set_title('Comparação: Espectro Médio Antes/Depois da Filtragem', 
                 fontsize=12, fontweight='bold')
    ax2.legend(loc='best', fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    plt.suptitle('Etapa 4: Filtragem de Bandas Ruidosas', 
                fontsize=14, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    plot_path = os.path.join(output_dir, f"{folder_name}_04_filtragem.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   📊 Gráfico salvo: {plot_path}")


# ============================================================================
# PIPELINE COMPLETO
# ============================================================================
def run_preprocessing_pipeline(output_dir: str, folder_name: str):
    """
    Executa todo o pipeline de pré-processamento espectral:
    1. MSC (Multiplicative Signal Correction)
    2. Log(1/R) (Transformação logarítmica)
    3. Derivada de Savitzky-Golay (1ª ordem)
    4. Filtragem de bandas ruidosas
    """
    print("\n" + "="*70)
    print("🚀 INICIANDO PIPELINE DE PRÉ-PROCESSAMENTO ESPECTRAL")
    print("="*70)
    print(f"📁 Diretório: {output_dir}")
    print(f"📝 Prefixo dos arquivos: {folder_name}")
    print(f"🗂️  Manter intermediários: {KEEP_INTERMEDIATE}")
    print(f"📊 Gerar gráficos: {GENERATE_PLOTS}")
    
    # Criar diretório se não existir
    os.makedirs(output_dir, exist_ok=True)
    
    # ETAPA 1: MSC
    df_msc_before, df_msc_after = apply_msc(output_dir, folder_name)
    if GENERATE_PLOTS:
        plot_msc_comparison(df_msc_before, df_msc_after, output_dir, folder_name)
    
    # ETAPA 2: Log(1/R)
    df_log_before, df_log_after = apply_log_transform(output_dir, folder_name)
    if GENERATE_PLOTS:
        plot_log_comparison(df_log_before, df_log_after, output_dir, folder_name)
    
    # ETAPA 3: Derivada
    df_deriv_before, df_deriv_after = apply_derivative(output_dir, folder_name)
    if GENERATE_PLOTS:
        plot_derivative_comparison(df_deriv_before, df_deriv_after, output_dir, folder_name)
    
    # ETAPA 4: Filtragem
    df_filt_before, df_filt_after, bandas_rem = apply_noise_filtering(output_dir, folder_name)
    if GENERATE_PLOTS:
        plot_filtering_analysis(df_filt_before, df_filt_after, bandas_rem, output_dir, folder_name)
    
    print("\n" + "="*70)
    print("✅ PIPELINE CONCLUÍDO COM SUCESSO!")
    print("="*70)
    print(f"📊 Arquivo final: {folder_name}_filtrado.csv")
    print(f"📊 Dimensões finais: {df_filt_after.shape}")
    if GENERATE_PLOTS:
        print(f"📊 Gráficos salvos: 4 arquivos PNG gerados")
    print("="*70 + "\n")


# ============================================================================
# EXEMPLO DE USO
# ============================================================================
if __name__ == "__main__":
    # Configurar parâmetros
    OUTPUT_DIR = "./resultados"
    FOLDER_NAME = "amostra_2024"
    
    # Executar pipeline
    run_preprocessing_pipeline(OUTPUT_DIR, FOLDER_NAME)
    
    # Exemplo de uso com múltiplas amostras:
    # for